# Phase 4 · Court Judgment Prediction (CJPE)

**Task:** predict the binary outcome (0/1) of a case from its judgment text. **Metric: macro-F1** over both classes (so the minority outcome matters). Output: `[{"id": …, "pred": 0|1}, …]`.

**Requirements (§10):** start from the Phase 1 models and use all four. At least 6 substantially different experiments: at least 5 fine-tuning techniques from {full, partial / layer-wise, layer freezing / unfreezing, BitFit, **LoRMA (compulsory)**, AdaLoRA, adapters} plus at least 1 **KD**. Required analyses: long-document representation (chunking / hierarchical / section selection), which parts of the judgment matter, class imbalance, false positives vs false negatives, and model size versus inference cost. **No external legal corpora or pretrained models.**

**P4-0:** download and profile the data: sizes, label balance, document lengths in our tokens, how texts start and end, and duplicates across splits.

In [1]:
# ============================================================================================
# P4-0 — Phase 4 data download + profile (CPU only, a few minutes)
# ============================================================================================
import os, sys, json, glob, hashlib, random
import numpy as np
from collections import Counter
from huggingface_hub import snapshot_download
from kaggle_secrets import UserSecretsClient

WORK  = "/kaggle/working"
TOKEN = UserSecretsClient().get_secret("HF_WRITE_TOKEN")
sys.path.insert(0, f"{WORK}/src")

# ---- 1. Phase-1 assets present? (models + tokenizer are needed for every phase) ----------------
need = [f"{WORK}/tokenizer/hf_a1_bpe_v2_32k.json"] + [f"{WORK}/checkpoints/{m}/best.pt" for m in
        ["m1_dense_100m", "m2_mla_moe", "m3_encdec", "m4_copy"]]
print("phase-1 assets OK:", all(os.path.exists(p) for p in need), "| missing:",
      [os.path.relpath(p, WORK) for p in need if not os.path.exists(p)])

# ---- 2. download Phase-4 data --------------------------------------------------------------------
snapshot_download("Exploration-Lab/CS-781-Capstone", repo_type="dataset", token=TOKEN,
                  allow_patterns=["Phase_4_CJPE/**"], local_dir=f"{WORK}/data_p4")
files = sorted(glob.glob(f"{WORK}/data_p4/**/*.json*", recursive=True))
for f in files:
    print(f"   {os.path.relpath(f, WORK)}  {os.path.getsize(f)/1e6:.1f} MB")

def load(path):
    """list of records or dict id -> record; jsonl also accepted."""
    if path.endswith(".jsonl"):
        return [json.loads(l) for l in open(path)]
    obj = json.load(open(path))
    if isinstance(obj, dict):
        obj = [{"id": k, **v} if isinstance(v, dict) else {"id": k, "text": v} for k, v in obj.items()]
    return obj

S = {os.path.basename(f).split(".")[0]: load(f) for f in files}
for name, recs in S.items():
    print(f"\n=== {name}: {len(recs)} records | keys {list(recs[0].keys())}")
    print("    example:", {k: (v[:120] + "…" if isinstance(v, str) and len(v) > 120 else v) for k, v in recs[0].items()})

# ---- 3. labels and lengths -------------------------------------------------------------------------
LABEL_KEY = next((k for k in ("label", "labels", "pred", "y", "decision") if k in S["train"][0]), None)
print(f"\nlabel field: {LABEL_KEY!r}")
from tokenizers import Tokenizer
from common import normalise
tk = Tokenizer.from_file(f"{WORK}/tokenizer/hf_a1_bpe_v2_32k.json")
random.seed(0)
for name, recs in S.items():
    chars = np.array([len(r["text"]) for r in recs])
    samp = random.sample(recs, min(500, len(recs)))                        # token counts on a sample
    toks = np.array([len(tk.encode(normalise(r["text"]), add_special_tokens=False).ids) for r in samp])
    line = (f"\n=== {name}: chars median {int(np.median(chars)):,} p90 {int(np.percentile(chars, 90)):,} "
            f"max {chars.max():,} | tokens (sample of {len(samp)}) median {int(np.median(toks)):,} "
            f"p90 {int(np.percentile(toks, 90)):,} max {toks.max():,} | ≤512: {np.mean(toks <= 512):.0%} "
            f"≤2048: {np.mean(toks <= 2048):.0%} | chars/token {chars.mean() / max(1, toks.mean() * len(recs) / len(samp) * len(samp) / len(recs)):.1f}")
    print(line)
    if LABEL_KEY and LABEL_KEY in recs[0]:
        c = Counter(r[LABEL_KEY] for r in recs)
        print(f"    labels: {dict(c)}  ({', '.join(f'{k}: {v/len(recs):.1%}' for k, v in sorted(c.items()))})")
    print(f"    total tokens (estimated): {np.mean(toks) * len(recs) / 1e6:.0f}M")

# ---- 4. how do documents start and end? (the end is usually where the decision is written) -----
for r in random.sample(S["train"], 2):
    t = r["text"]
    print(f"\n--- train id {r['id']} | label {r.get(LABEL_KEY)} | {len(t):,} chars")
    print("   START:", repr(t[:300]))
    print("   END:  ", repr(t[-400:]))

# ---- 5. duplicates / overlap between splits (ids and exact text) ---------------------------------
H = {n: {hashlib.md5(r["text"].encode()).hexdigest() for r in recs} for n, recs in S.items()}
I = {n: {str(r["id"]) for r in recs} for n, recs in S.items()}
names = list(S)
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        a, b = names[i], names[j]
        print(f"overlap {a} vs {b}: ids {len(I[a] & I[b])} | identical texts {len(H[a] & H[b])}")
print("duplicate texts inside train:", len(S["train"]) - len(H["train"]))

phase-1 assets OK: True | missing: []


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

   data_p4/Phase_4_CJPE/Test/test.json  35.3 MB
   data_p4/Phase_4_CJPE/Train/dev.json  80.3 MB
   data_p4/Phase_4_CJPE/Train/train.json  736.6 MB

=== test: 1517 records | keys ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']
    example: {'id': '1986_397', 'text': 'civil appellate jurisdiction civil appeal number 1257\n nl  of 1985.\nfrom the judgment and order dated 24th january 1985 …', 'label': 1, 'expert_1': None, 'expert_2': None, 'expert_3': None, 'expert_4': None, 'expert_5': None}

=== dev: 3505 records | keys ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']
    example: {'id': '1989_75', 'text': 'civil appellate jurisdiction civil appeal number 1709\nof\n1988.\nfrom the judgment and order dated 9.10.1987 of t\nhe\ncentr…', 'label': 0, 'expert_1': None, 'expert_2': None, 'expert_3': None, 'expert_4': None, 'expert_5': None}

=== train: 37387 records | keys ['id', 'text', 'label', 'expert_1', 'expert_2', 'ex

## P4-1 · Clean splits + token cache

**Data findings (P4-0):** 37,387 training cases (about 141M tokens; median about 2.7k tokens per case, so only 8% fit in 512 tokens); train is 59 / 41, while dev and test are 50 / 50 (**prior shift**, so the threshold is calibrated on dev); 5,710 duplicate texts in train; the final decision has been removed from the texts (they end in the court's reasoning).

**Integrity:** all 1,517 test cases also appear in dev.json (identical texts), and test.json carries a `label` field. We **never use test labels**, and we **remove those 1,517 cases from dev**. Model selection and threshold calibration use the remaining **1,988 clean dev cases** only. The duplication was reported to the TAs.

**This cell (CPU):** deduplicate train (dropping duplicate groups with conflicting labels), build clean dev, and tokenise every case **once** with our A1-BPE tokenizer (4 processes). Token ids are stored as uint16 arrays, so every input strategy (first / middle / last k tokens, all chunks) is cheap to build later.

In [1]:
# ============================================================================================
# P4-1 — Clean splits (dedup train, test-free dev) + one-time tokenisation cache (CPU, 4 procs)
#   Writes data_p4/clean/{train,dev,test}.json (id, label) and data_p4/tok/{split}.npz (all tokens)
# ============================================================================================
import os, sys, json, hashlib, time
import numpy as np
from collections import Counter, defaultdict
from multiprocessing import Pool

WORK = "/kaggle/working"; sys.path.insert(0, f"{WORK}/src")
RAW = f"{WORK}/data_p4/Phase_4_CJPE"
OUTC, OUTT = f"{WORK}/data_p4/clean", f"{WORK}/data_p4/tok"
os.makedirs(OUTC, exist_ok=True); os.makedirs(OUTT, exist_ok=True)

train = json.load(open(f"{RAW}/Train/train.json"))
dev   = json.load(open(f"{RAW}/Train/dev.json"))
test  = json.load(open(f"{RAW}/Test/test.json"))
h = lambda t: hashlib.md5(t.encode()).hexdigest()

# ---- 1. dev vs test: how big is the overlap, and do the labels agree? (diagnostic only) -------
test_ids = {str(r["id"]) for r in test}
dev_lab = {str(r["id"]): r["label"] for r in dev}
agree = np.mean([dev_lab[i] == r["label"] for r in test for i in [str(r["id"])] if i in dev_lab])
print(f"test cases inside dev: {sum(i in dev_lab for i in test_ids)}/{len(test_ids)} | "
      f"dev label == test.json label on them: {agree:.1%}  (diagnostic only — never used)")

# ---- 2. clean dev = dev without any test case (by id AND by text) ----------------------------
test_h = {h(r["text"]) for r in test}
dev_clean = [r for r in dev if str(r["id"]) not in test_ids and h(r["text"]) not in test_h]
print(f"dev: {len(dev)} -> clean dev {len(dev_clean)} | labels {dict(Counter(r['label'] for r in dev_clean))}")

# ---- 3. dedup train; drop duplicate groups whose labels disagree; drop any train case seen in dev/test
groups = defaultdict(list)
for r in train:
    groups[h(r["text"])].append(r)
banned = test_h | {h(r["text"]) for r in dev}
train_clean, conflicts = [], 0
for k, rs in groups.items():
    if k in banned:
        continue
    if len({r["label"] for r in rs}) > 1:
        conflicts += 1; continue
    train_clean.append(rs[0])
print(f"train: {len(train)} -> {len(train_clean)} unique cases | conflicting-label groups dropped: {conflicts} | "
      f"labels {dict(Counter(r['label'] for r in train_clean))}")

# ---- 4. tokenise every case once (normalise + A1-BPE), 4 worker processes ----------------------
def _init():
    global _tk, _norm
    from tokenizers import Tokenizer
    from common import normalise as _n
    _tk = Tokenizer.from_file(f"{WORK}/tokenizer/hf_a1_bpe_v2_32k.json"); _norm = _n

def _enc(text):
    return np.asarray(_tk.encode(_norm(text), add_special_tokens=False).ids, dtype=np.uint16)

def cache(name, recs):
    t0 = time.time()
    with Pool(4, initializer=_init) as pool:
        arrs = pool.map(_enc, [r["text"] for r in recs], chunksize=64)
    lens = np.array([len(a) for a in arrs], dtype=np.int64)
    np.savez(f"{OUTT}/{name}.npz", ids=np.concatenate(arrs), offsets=np.concatenate([[0], np.cumsum(lens)]),
             labels=np.array([r["label"] for r in recs] if name != "test" else [-1] * len(recs)))
    json.dump([{"id": str(r["id"]), "label": (r["label"] if name != "test" else None)} for r in recs],
              open(f"{OUTC}/{name}.json", "w"))
    print(f"   {name}: {len(recs)} cases, {lens.sum()/1e6:.1f}M tokens | median {int(np.median(lens))} "
          f"p90 {int(np.percentile(lens, 90))} | {time.time()-t0:.0f}s")

print("\ntokenising (CPU) ...")
cache("dev", dev_clean)
cache("test", test)                       # test labels are deliberately NOT stored
cache("train", train_clean)
import shutil; print(f"free disk: {shutil.disk_usage(WORK).free/1e9:.1f} GB")

test cases inside dev: 1517/1517 | dev label == test.json label on them: 100.0%  (diagnostic only — never used)
dev: 3505 -> clean dev 1988 | labels {0: 994, 1: 994}
train: 37387 -> 31677 unique cases | conflicting-label groups dropped: 0 | labels {0: 18552, 1: 13125}

tokenising (CPU) ...
   dev: 1988 cases, 8.9M tokens | median 3408 p90 8318 | 15s
   test: 1517 cases, 7.0M tokens | median 3532 p90 8557 | 12s
   train: 31677 cases, 121.0M tokens | median 2465 p90 7767 | 203s
free disk: 8.5 GB


## P4-2 · Which part of a judgment predicts the outcome? (section study)

**Clean data (P4-1):** train 31,677 unique cases (121M tokens, 59 / 41); clean dev 1,988 cases (50 / 50). The test labels in test.json agree 100% with the dev labels of the same cases, which confirms the leak; those cases are excluded everywhere.

**Model:** a Phase 1 backbone with bidirectional attention → masked mean pooling → linear layer to 2 classes; balanced class-weighted CE (corrects the 59 / 41 training prior); macro-F1 reported at threshold 0.5 and at a threshold tuned on clean dev.

**Section study:** M1 + LoRA (r = 8, q and v), 1 epoch, the same 512-token budget for four different views of every case: **first 512**, **middle 512**, **last 512**, and **head + tail** (first 128 + last 384). Because the final decision was removed from the texts, we expect the end (the court's reasoning) to be the most informative part.

In [2]:
%%writefile /kaggle/working/src/p4_cjpe.py
"""
p4_cjpe.py -- Phase 4 (Court Judgment Prediction, binary) on top of ANY Phase-1 backbone.

Data comes from the token cache written by P4-1 (data_p4/tok/{split}.npz: all tokens of every case,
uint16, plus offsets + labels; test labels are stored as -1 and never used).

Input views of a long case (our A1-BPE tokens):
  section(arr, "last",  k)  last k tokens           (the decision itself was removed from the texts,
  section(arr, "first", k)  first k tokens           so the reasoning at the end is expected to matter most)
  section(arr, "middle",k)  k tokens around the centre
  section(arr, "headtail",k) first k/4 + last 3k/4
DocClassifier: backbone (bidirectional) -> masked mean pool -> Linear(d, 2).
Any adaptation method from p3_rr.apply_method_p3 / p2_peft.apply_method:
  full, partial (top-k layers), normgain (BitFit analogue), lora, rslora, lorma_plus, lorma_pi, ia3,
  adapter, prompt.
Class-weighted CE (train is 59/41, dev/test are 50/50). Decision threshold reported at 0.5 and tuned on dev.
"""
from __future__ import annotations

import contextlib
import math
import random
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import p2_model as pm
import p2_peft as pp
import p3_rr as rr

PAD_ID = 0


# ----------------------------------------------------------------------------- data
class Split:
    """All tokens of one split: arr(i) -> np.uint16 token ids of case i."""
    def __init__(self, path):
        z = np.load(path)
        self.ids, self.off, self.labels = z["ids"], z["offsets"], z["labels"]
        self.n = len(self.labels)

    def arr(self, i):
        return self.ids[self.off[i]:self.off[i + 1]]


def section(a, mode="last", k=512):
    n = len(a)
    if n <= k:
        return a
    if mode == "last":
        return a[n - k:]
    if mode == "first":
        return a[:k]
    if mode == "middle":
        s = (n - k) // 2
        return a[s:s + k]
    if mode == "headtail":
        h = k // 4
        return np.concatenate([a[:h], a[n - (k - h):]])
    raise ValueError(mode)


def collate(seqs, device):
    L = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), L), PAD_ID, dtype=torch.long)
    mask = torch.zeros((len(seqs), L), dtype=torch.bool)
    for b, s in enumerate(seqs):
        ids[b, :len(s)] = torch.from_numpy(s.astype(np.int64)); mask[b, :len(s)] = True
    return ids.to(device), mask.to(device)


def _autocast(device):
    return (torch.autocast("cuda", dtype=torch.float16) if str(device).startswith("cuda")
            else contextlib.nullcontext())


# ----------------------------------------------------------------------------- model
class DocClassifier(nn.Module):
    def __init__(self, tagger, dropout=0.1):
        super().__init__()
        self.tagger = tagger
        self.drop = nn.Dropout(dropout)

    def embed(self, ids, mask):
        """Masked mean of the final hidden states, (B, d) in fp32."""
        tg = self.tagger
        ctx = pm.full_attention() if (tg.bidirectional and tg.arch != "encdec") else contextlib.nullcontext()
        with ctx:
            h = tg.hidden(ids)
        if isinstance(tg.head, pp.DropPrefix):                         # prompt tuning: drop soft prompt
            h = h[:, tg.head.n:]
        h = h.float(); m = mask.unsqueeze(-1).float()
        return (h * m).sum(1) / m.sum(1).clamp(min=1)

    def head(self):
        h = self.tagger.head
        return h.head if isinstance(h, pp.DropPrefix) else h

    def forward(self, ids, mask):
        return self.head()(self.drop(self.embed(ids, mask)))           # (B, 2)


def build(ckpt, method="lora", bidirectional=True, partial_k=4, **kw):
    tg = pm.build_tagger(ckpt, bidirectional=bidirectional)
    tg.head = nn.Linear(tg.backbone.cfg.d_model, 2)
    nn.init.normal_(tg.head.weight, 0.0, 0.02); nn.init.zeros_(tg.head.bias)
    if method == "partial":
        pp.apply_method(tg, "partial", top_k=partial_k)
    else:
        rr.apply_method_p3(tg, method, **kw)
    for p in tg.head.parameters():
        p.requires_grad = True
    return DocClassifier(tg)


# ----------------------------------------------------------------------------- inference + metrics
@torch.no_grad()
def predict(model, split, mode="last", k=512, device="cuda", bs=32):
    """P(label = 1) for every case of the split."""
    model.eval(); out = np.zeros(split.n, dtype=np.float32)
    order = np.argsort([min(k, split.off[i + 1] - split.off[i]) for i in range(split.n)])
    for b in range(0, split.n, bs):
        idx = order[b:b + bs]
        ids, mask = collate([section(split.arr(i), mode, k) for i in idx], device)
        with _autocast(device):
            p = torch.softmax(model(ids, mask).float(), -1)[:, 1]
        out[idx] = p.cpu().numpy()
    return out


def metrics(y, p, t=0.5):
    from sklearn.metrics import f1_score, precision_score, recall_score
    yh = (p >= t).astype(int)
    return {"macro_f1": float(f1_score(y, yh, average="macro")),
            "macro_p": float(precision_score(y, yh, average="macro", zero_division=0)),
            "macro_r": float(recall_score(y, yh, average="macro", zero_division=0)),
            "acc": float((y == yh).mean()), "pos_rate": float(yh.mean()), "t": float(t)}


def tune_threshold(y, p):
    ts = np.linspace(0.05, 0.95, 91)
    f = [metrics(y, p, t)["macro_f1"] for t in ts]
    return float(ts[int(np.argmax(f))])


# ----------------------------------------------------------------------------- training
def train(model, tr, dev, mode="last", k=512, epochs=1, lr=5e-4, head_lr=1e-3, bs=8, device="cuda",
          seed=0, warmup=0.03, evals_per_epoch=2, max_train=None, layerwise_unfreeze=False,
          soft=None, kd_T=2.0, kd_lam=0.5, log=print):
    """Class-weighted CE (+ optional KD on teacher probs `soft`), AdamW, warm-up + linear decay, fp16.
    Evaluates on dev `evals_per_epoch` times per epoch, keeps the best macro-F1 (threshold 0.5).
    layerwise_unfreeze: gradual unfreezing (top layer first, one more layer every 1/L of training)."""
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)
    model.to(device)
    idx_all = np.arange(tr.n)
    if max_train:
        idx_all = np.random.RandomState(seed).permutation(tr.n)[:max_train]
    y = tr.labels
    cnt = np.bincount(y[idx_all], minlength=2).astype(float)
    w = torch.tensor(cnt.sum() / (2 * cnt), dtype=torch.float, device=device)   # balanced class weights
    layers = list(pp._layers(model.tagger))
    if layerwise_unfreeze:                                            # start: only head trainable
        for p in model.tagger.backbone.parameters():
            p.requires_grad = False
    params = [p for p in model.parameters() if p.requires_grad or layerwise_unfreeze]
    head_ids = {id(p) for p in model.head().parameters()}
    opt = torch.optim.AdamW([{"params": [p for p in model.parameters() if id(p) not in head_ids
                                         and (p.requires_grad or layerwise_unfreeze)], "lr": lr},
                             {"params": list(model.head().parameters()), "lr": head_lr}], weight_decay=0.01)
    steps = epochs * math.ceil(len(idx_all) / bs); warm = max(1, int(warmup * steps))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min((s + 1) / warm, max(0.0, (steps - s) / max(1, steps - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=str(device).startswith("cuda"))
    eval_at = {int(steps * (j + 1) / (epochs * evals_per_epoch)) for j in range(epochs * evals_per_epoch)}
    best, best_state, hist, step, t0, run_loss = -1.0, None, [], 0, time.time(), 0.0
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    log(f"trainable {n_tr/1e6:.3f}M | {len(idx_all)} cases | {steps} steps | view {mode}-{k}"
        + (" | gradual unfreezing" if layerwise_unfreeze else "") + (" | KD" if soft is not None else ""))
    for ep in range(epochs):
        order = np.random.permutation(idx_all)
        for b in range(0, len(order), bs):
            if layerwise_unfreeze:                                    # unfreeze one more top layer on schedule
                n_on = min(len(layers), 1 + int(step / max(1, steps) * len(layers)))
                for L in layers[len(layers) - n_on:]:
                    for p in L.parameters():
                        p.requires_grad = True
            model.train()
            bi = order[b:b + bs]
            ids, mask = collate([section(tr.arr(i), mode, k) for i in bi], device)
            lab = torch.tensor(y[bi], dtype=torch.long, device=device)
            with _autocast(device):
                logits = model(ids, mask).float()
            loss = F.cross_entropy(logits, lab, weight=w)
            if soft is not None:                                      # KD on teacher P(1)
                tp = torch.tensor(soft[bi], dtype=torch.float, device=device).clamp(1e-4, 1 - 1e-4)
                t_log = torch.stack([torch.log1p(-tp), torch.log(tp)], -1)
                kl = F.kl_div(F.log_softmax(logits / kd_T, -1), F.log_softmax(t_log / kd_T, -1),
                              log_target=True, reduction="batchmean") * kd_T * kd_T
                loss = (1 - kd_lam) * loss + kd_lam * kl
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            scaler.step(opt); scaler.update(); sched.step(); step += 1; run_loss += loss.item()
            if step % 500 == 0:
                log(f"   step {step}/{steps} | loss {run_loss/500:.4f} | {time.time()-t0:.0f}s"); run_loss = 0.0
            if step in eval_at:
                p = predict(model, dev, mode, k, device)
                m = metrics(dev.labels, p); m_t = metrics(dev.labels, p, tune_threshold(dev.labels, p))
                hist.append({"step": step, "f1@0.5": m["macro_f1"], "f1@tuned": m_t["macro_f1"], "t": m_t["t"],
                             "pos_rate": m["pos_rate"]})
                log(f"   EVAL step {step}: dev macro-F1 {m['macro_f1']:.4f} @0.5 (pos rate {m['pos_rate']:.2f}) | "
                    f"{m_t['macro_f1']:.4f} @t={m_t['t']:.2f} | {time.time()-t0:.0f}s")
                if m["macro_f1"] > best:
                    best = m["macro_f1"]
                    best_state = {n: q.detach().cpu().clone() for n, q in model.named_parameters()
                                  if q.requires_grad or layerwise_unfreeze}   # gradual unfreezing: keep all
    model.load_state_dict({k_: v.to(device) for k_, v in best_state.items()}, strict=False)
    return hist, best


# ----------------------------------------------------------------------------- hierarchical (chunk) view
@torch.no_grad()
def encode_chunks(model, split, chunk=512, device="cuda", bs=32, max_chunks=64):
    """Mean-pooled embedding of EVERY chunk of every case (the last `max_chunks` chunks for very long
    cases). Returns list of (n_chunks, d) float16 arrays."""
    model.eval()
    items = []                                                         # (case, chunk index, start)
    for i in range(split.n):
        n = split.off[i + 1] - split.off[i]
        starts = list(range(0, max(1, n), chunk))[-max_chunks:]
        items += [(i, j, s) for j, s in enumerate(starts)]
    out = [None] * split.n
    tmp = {}
    for b in range(0, len(items), bs):
        batch = items[b:b + bs]
        ids, mask = collate([split.arr(i)[s:s + chunk] for i, _, s in batch], device)
        with _autocast(device):
            e = model.embed(ids, mask).half().cpu().numpy()
        for (i, j, _), v in zip(batch, e):
            tmp.setdefault(i, []).append((j, v))
    for i in range(split.n):
        out[i] = np.stack([v for _, v in sorted(tmp[i], key=lambda x: x[0])])
    return out


class ChunkPooler(nn.Module):
    """Hierarchical head over a case's chunk embeddings: BiGRU + attention pooling -> 2 logits.
    The attention weights tell which parts of the judgment drive the prediction."""
    def __init__(self, d=768, h=256, dropout=0.2):
        super().__init__()
        self.inp = nn.Sequential(nn.LayerNorm(d), nn.Dropout(dropout))
        self.gru = nn.GRU(d, h, batch_first=True, bidirectional=True)
        self.att = nn.Linear(2 * h, 1)
        self.out = nn.Sequential(nn.Dropout(dropout), nn.Linear(2 * h, 2))

    def forward(self, x, mask, return_att=False):                      # x (B, C, d), mask (B, C)
        z, _ = self.gru(self.inp(x))
        a = self.att(z).squeeze(-1).masked_fill(~mask, -1e4).softmax(-1)
        o = self.out((a.unsqueeze(-1) * z).sum(1))
        return (o, a) if return_att else o

Writing /kaggle/working/src/p4_cjpe.py


In [3]:
%%writefile /kaggle/working/src/p4_run.py
"""
p4_run.py -- background runner for Phase-4 (CJPE) experiments, one process per GPU:

    CUDA_VISIBLE_DEVICES=0 PYTHONUNBUFFERED=1 setsid nohup python p4_run.py --plan plan.json >> log 2>&1 &

plan.json = list of jobs. Job types:
  run  (default): {"name", "ckpt": "m1".."m4", "method", "mode": "last", "k": 512, "epochs", "lr",
                   "bs", "seed", "max_train", "layerwise_unfreeze", "partial_k", method kwargs...,
                   KD: "teachers": [run names], "student_layers": [...], "kd_T", "kd_lam"}
  prob:  {"type": "prob", "name": X, "train": true}  re-load run X and save P(1) for train/dev/test
Outputs per run: stats/p4_runs/<name>.json (written last = done marker), p4_ckpt/<name>.pt (trained
params), p4_prob/<name>.pkl ({"dev": P(1), "test": P(1)[, "train"]}).
Model selection / thresholds use the CLEAN dev only (test cases removed); test labels are never read.
"""
import argparse
import json
import os
import pickle
import sys
import time
import traceback

import numpy as np
import torch

WORK = "/kaggle/working"
sys.path.insert(0, f"{WORK}/src")
import p4_cjpe as cj                                                  # noqa: E402
import p2_peft as pp                                                  # noqa: E402
import p3_rr as rr                                                    # noqa: E402

ap = argparse.ArgumentParser(); ap.add_argument("--plan", required=True)
PLAN = json.load(open(ap.parse_args().plan))
DEV = "cuda:0"
CK = {"m1": f"{WORK}/checkpoints/m1_dense_100m/best.pt", "m2": f"{WORK}/checkpoints/m2_mla_moe/best.pt",
      "m3": f"{WORK}/checkpoints/m3_encdec/best.pt", "m4": f"{WORK}/checkpoints/m4_copy/best.pt"}
OUT, PRB, CKD = f"{WORK}/stats/p4_runs", f"{WORK}/p4_prob", f"{WORK}/p4_ckpt"
for d in (OUT, PRB, CKD):
    os.makedirs(d, exist_ok=True)


def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)


TR, DV, TE = (cj.Split(f"{WORK}/data_p4/tok/{s}.npz") for s in ("train", "dev", "test"))
log(f"data: train {TR.n} | clean dev {DV.n} | test {TE.n}")


def make_model(ckpt, method, student_layers=None, partial_k=4, **kw):
    m = cj.build(CK[ckpt], method, bidirectional=True, partial_k=partial_k, **kw)
    if student_layers:
        pp.shrink_dense(m.tagger, tuple(student_layers))
    return m


def run(name, ckpt="m1", method="lora", mode="last", k=512, epochs=1, lr=5e-4, bs=8, seed=0,
        max_train=None, layerwise_unfreeze=False, partial_k=4, teachers=None, student_layers=None,
        kd_T=2.0, kd_lam=0.5, **kw):
    t0 = time.time()
    model = make_model(ckpt, method, student_layers, partial_k, **kw)
    soft = None
    if teachers:                                                       # mean teacher P(1) on train cases
        soft = np.mean([pickle.load(open(f"{PRB}/{t}.pkl", "rb"))["train"] for t in teachers], 0)
    log(f"=== {name}: {ckpt} | {method} | {mode}-{k} | epochs {epochs} lr {lr}"
        + (f" | student layers {student_layers} | teachers {teachers}" if teachers or student_layers else ""))
    hist, best = cj.train(model, TR, DV, mode, k, epochs, lr, bs=bs, device=DEV, seed=seed,
                          max_train=max_train, layerwise_unfreeze=layerwise_unfreeze,
                          soft=soft, kd_T=kd_T, kd_lam=kd_lam, log=log)
    pd = cj.predict(model, DV, mode, k, DEV)
    torch.cuda.synchronize(); ti = time.time()
    pt = cj.predict(model, TE, mode, k, DEV)
    torch.cuda.synchronize(); infer = round(time.time() - ti, 1)
    t = cj.tune_threshold(DV.labels, pd)
    out = {"cfg": dict(ckpt=ckpt, method=method, mode=mode, k=k, epochs=epochs, lr=lr, bs=bs, seed=seed,
                       max_train=max_train, layerwise_unfreeze=layerwise_unfreeze, partial_k=partial_k,
                       teachers=teachers, student_layers=student_layers, kd_T=kd_T, kd_lam=kd_lam, **kw),
           "trainable_M": round(sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6, 3),
           "total_M": round(sum(p.numel() for p in model.parameters()) / 1e6, 2),
           "infer_test_s": infer, "hist": hist, "min": round((time.time() - t0) / 60, 1),
           "dev@0.5": cj.metrics(DV.labels, pd), "dev@tuned": cj.metrics(DV.labels, pd, t),
           "test_pos_rate@tuned": float((pt >= t).mean())}
    torch.save({n: p.detach().half().cpu() for n, p in model.named_parameters()
                if p.requires_grad or layerwise_unfreeze or method in ("full", "partial")}, f"{CKD}/{name}.pt")
    pickle.dump({"dev": pd, "test": pt}, open(f"{PRB}/{name}.pkl", "wb"))
    json.dump(out, open(f"{OUT}/{name}.json", "w"), indent=1)
    log(f"DONE {name}: dev macro-F1 {out['dev@0.5']['macro_f1']:.4f} @0.5 | {out['dev@tuned']['macro_f1']:.4f} "
        f"@t={t:.2f} | test pos-rate {out['test_pos_rate@tuned']:.2f} | {out['min']} min")
    del model; torch.cuda.empty_cache()


def prob_job(name, train=False):
    R = json.load(open(f"{OUT}/{name}.json")); c = {k: v for k, v in R["cfg"].items() if v is not None}
    mode, k = c.pop("mode"), c.pop("k")
    for key in ("epochs", "lr", "bs", "seed", "max_train", "layerwise_unfreeze", "teachers", "kd_T", "kd_lam"):
        c.pop(key, None)
    m = make_model(c.pop("ckpt"), c.pop("method"), c.pop("student_layers", None), c.pop("partial_k", 4), **c)
    sd = {n: v.float() for n, v in torch.load(f"{CKD}/{name}.pt", map_location="cpu").items()}
    m.load_state_dict(sd, strict=False); m.to(DEV)
    path = f"{PRB}/{name}.pkl"; out = pickle.load(open(path, "rb"))
    if train and "train" not in out:
        ti = time.time(); out["train"] = cj.predict(m, TR, mode, k, DEV)
        log(f"   {name}: train probs in {time.time()-ti:.0f}s")
    pickle.dump(out, open(path, "wb"))
    chk = cj.metrics(DV.labels, cj.predict(m, DV, mode, k, DEV))["macro_f1"]
    log(f"PROB {name}: dev re-check {chk:.4f} (should equal its @0.5 score)")
    del m; torch.cuda.empty_cache()


for job in PLAN:
    job = dict(job); name = job.pop("name"); typ = job.pop("type", "run")
    try:
        if typ == "prob":
            prob_job(name, job.get("train", False))
        elif os.path.exists(f"{OUT}/{name}.json"):
            log(f"skip {name} (already done)")
        else:
            run(name, **job)
    except Exception:
        log(f"!!! {name} FAILED"); traceback.print_exc(); torch.cuda.empty_cache()
log("PLAN DONE")

Writing /kaggle/working/src/p4_run.py


In [1]:
# ============================================================================================
# P4-2L — Section study in the background: M1 + LoRA, 1 epoch, 4 views x 512 tokens (2 per GPU)
#   Refuses to start if a p4_run.py job is already running; finished runs are skipped on re-launch.
# ============================================================================================
import json, subprocess
WORK = "/kaggle/working"
BASE = dict(ckpt="m1", method="lora", k=512, epochs=1, lr=5e-4, bs=8)
PLANS = {0: [dict(name="m1_lora_last512",     mode="last",     **BASE),
             dict(name="m1_lora_first512",    mode="first",    **BASE)],
         1: [dict(name="m1_lora_middle512",   mode="middle",   **BASE),
             dict(name="m1_lora_headtail512", mode="headtail", **BASE)]}
if subprocess.run("pgrep -f '[p]4_run.py'", shell=True, capture_output=True, text=True).stdout.strip():
    print("p4_run.py already running -> not launching again")
else:
    for g, plan in PLANS.items():
        pf = f"{WORK}/stats/p4_plan2_gpu{g}.json"; json.dump(plan, open(pf, "w"))
        subprocess.run(f"cd {WORK}/src && CUDA_VISIBLE_DEVICES={g} PYTHONUNBUFFERED=1 setsid nohup "
                       f"python p4_run.py --plan {pf} >> {WORK}/stats/p4_gpu{g}.log 2>&1 < /dev/null &", shell=True)
    print("launched section study on both GPUs -> watch with P4-W")

launched section study on both GPUs -> watch with P4-W


In [1]:
# ============================================================================================
# P4-W — Phase-4 progress + results (reads logs and result files only)
# ============================================================================================
import json, glob, os, subprocess
WORK = "/kaggle/working"
print(subprocess.run("echo 'p4_run processes:' $(pgrep -fc '[p]4_run.py'); "
                     "nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout)
for g in (0, 1):
    p = f"{WORK}/stats/p4_gpu{g}.log"
    lines = open(p).read().splitlines() if os.path.exists(p) else []
    print(f"--- GPU {g}: key events")
    for l in lines:
        if any(k in l for k in ("===", "EVAL", "DONE", "PROB", "FAILED", "Error", "data:")):
            print("   ", l[:180])
    print(f"--- GPU {g}: last 2 lines"); [print("   ", l[:180]) for l in lines[-2:]]

R = {os.path.basename(f)[:-5]: json.load(open(f)) for f in glob.glob(f"{WORK}/stats/p4_runs/*.json")}
print(f"\n{'run':<24}{'trainable':>11}{'F1@0.5':>8}{'F1@t':>8}{'t':>6}{'P':>7}{'R':>7}{'acc':>7}{'test+':>7}{'test s':>8}{'min':>6}")
for n, r in sorted(R.items(), key=lambda kv: -kv[1]["dev@tuned"]["macro_f1"]):
    a, b = r["dev@0.5"], r["dev@tuned"]
    print(f"{n:<24}{r['trainable_M']:>10}M{a['macro_f1']:>8.4f}{b['macro_f1']:>8.4f}{b['t']:>6.2f}"
          f"{b['macro_p']:>7.3f}{b['macro_r']:>7.3f}{b['acc']:>7.3f}{r['test_pos_rate@tuned']:>7.2f}"
          f"{r['infer_test_s']:>8}{r['min']:>6}")

p4_run processes: 0
0, 0 %, 0 MiB
1, 0 %, 0 MiB

--- GPU 0: key events
    08:34:35 data: train 31677 | clean dev 1988 | test 1517
    08:34:38 === m1_lora_last512: m1 | lora | last-512 | epochs 1 lr 0.0005
    08:42:36    EVAL step 1980: dev macro-F1 0.6385 @0.5 (pos rate 0.65) | 0.6740 @t=0.62 | 475s
    08:50:33    EVAL step 3960: dev macro-F1 0.6715 @0.5 (pos rate 0.29) | 0.6963 @t=0.39 | 951s
    08:51:09 DONE m1_lora_last512: dev macro-F1 0.6715 @0.5 | 0.6963 @t=0.39 | test pos-rate 0.45 | 16.6 min
    08:51:12 === m1_lora_first512: m1 | lora | first-512 | epochs 1 lr 0.0005
    08:59:09    EVAL step 1980: dev macro-F1 0.4714 @0.5 (pos rate 0.84) | 0.5610 @t=0.67 | 477s
    09:07:05    EVAL step 3960: dev macro-F1 0.5515 @0.5 (pos rate 0.56) | 0.5794 @t=0.54 | 953s
    09:07:41 DONE m1_lora_first512: dev macro-F1 0.5515 @0.5 | 0.5794 @t=0.54 | test pos-rate 0.42 | 16.5 min
    09:07:41 PLAN DONE
--- GPU 0: last 2 lines
    09:07:41 DONE m1_lora_first512: dev macro-F1 0.5515 @0.5 

## P4-3 · Technique × architecture comparison (last-512 view)

**P4-2 result (M1 + LoRA, 1 epoch, 512 tokens):** last 512 **0.696** (tuned threshold) > head + tail 0.672 > middle 0.586 > first 512 0.579. The end of a judgment (the court's reasoning just before the removed decision) carries the outcome; the opening facts carry little. Threshold tuning adds about +0.025, because the predicted positive rate drifts during training (train prior 59 / 41 against dev 50 / 50).

**Runs:** last-512 view, 1 epoch each (budget-matched), balanced class weights, threshold tuned on clean dev. Techniques from §10.2.1: full FT, partial FT (top 4 layers), gradual layer unfreezing, BitFit (norm-gain analogue: our layers have no biases), **LoRMA+**, adapters, and all four architectures. Extra: a 1,024-token view (M1 + LoRA) to test whether more of the closing reasoning helps.

In [2]:
# ============================================================================================
# P4-3L — Technique x architecture runs (last-512, 1 epoch each), 5 per GPU, in the background
#   Finished runs are skipped on re-launch; watch with P4-W.
# ============================================================================================
import json, subprocess
WORK = "/kaggle/working"
B = dict(mode="last", k=512, epochs=1, bs=8)
PLANS = {
    0: [dict(name="m1_full_last512",      ckpt="m1", method="full",       lr=3e-5, **B),
        dict(name="m1_lorma_plus_last512", ckpt="m1", method="lorma_plus", lr=5e-4, **B),
        dict(name="m1_normgain_last512",  ckpt="m1", method="normgain",   lr=1e-3, **B),
        dict(name="m1_lora_last1024",     ckpt="m1", method="lora",       lr=5e-4, mode="last", k=1024, epochs=1, bs=8),
        dict(name="m3_full_last512",      ckpt="m3", method="full",       lr=5e-5, **B)],
    1: [dict(name="m1_partial4_last512",  ckpt="m1", method="partial",    lr=1e-4, partial_k=4, **B),
        dict(name="m1_unfreeze_last512",  ckpt="m1", method="full",       lr=5e-5, layerwise_unfreeze=True, **B),
        dict(name="m1_adapter_last512",   ckpt="m1", method="adapter",    lr=5e-4, **B),
        dict(name="m2_lorma_plus_last512", ckpt="m2", method="lorma_plus", lr=5e-4, **B),
        dict(name="m4_adapter_last512",   ckpt="m4", method="adapter",    lr=5e-4, **B)],
}
if subprocess.run("pgrep -f '[p]4_run.py'", shell=True, capture_output=True, text=True).stdout.strip():
    print("p4_run.py already running -> not launching again")
else:
    for g, plan in PLANS.items():
        pf = f"{WORK}/stats/p4_plan3_gpu{g}.json"; json.dump(plan, open(pf, "w"))
        subprocess.run(f"cd {WORK}/src && CUDA_VISIBLE_DEVICES={g} PYTHONUNBUFFERED=1 setsid nohup "
                       f"python p4_run.py --plan {pf} >> {WORK}/stats/p4_gpu{g}.log 2>&1 < /dev/null &", shell=True)
    print("launched P4-3 on both GPUs -> watch with P4-W")

launched P4-3 on both GPUs -> watch with P4-W


In [1]:
# ============================================================================================
# P4-W — Phase-4 progress + results (reads logs and result files only)
# ============================================================================================
import json, glob, os, subprocess
WORK = "/kaggle/working"

# ---- 1. background jobs alive? GPUs busy? ---------------------------------------------------------
print(subprocess.run("echo 'p4_run processes:' $(pgrep -fc '[p]4_run.py'); "
                     "nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout)

# ---- 2. per GPU: key events (run starts, evals, results, failures) + last lines ------------------
for g in (0, 1):
    p = f"{WORK}/stats/p4_gpu{g}.log"
    lines = open(p).read().splitlines() if os.path.exists(p) else []
    print(f"--- GPU {g}: key events")
    for l in lines:
        if any(k in l for k in ("===", "EVAL", "DONE", "PROB", "FAILED", "Error", "skip")):
            print("   ", l[:180])
    print(f"--- GPU {g}: last 2 lines")
    for l in lines[-2:]:
        print("   ", l[:180])

# ---- 3. results table, sorted by dev macro-F1 at the tuned threshold -----------------------------
R = {os.path.basename(f)[:-5]: json.load(open(f)) for f in glob.glob(f"{WORK}/stats/p4_runs/*.json")}
print(f"\n{'run':<24}{'trainable':>11}{'F1@0.5':>8}{'F1@t':>8}{'t':>6}{'P':>7}{'R':>7}{'acc':>7}"
      f"{'test+':>7}{'test s':>8}{'min':>6}")
for n, r in sorted(R.items(), key=lambda kv: -kv[1]["dev@tuned"]["macro_f1"]):
    a, b = r["dev@0.5"], r["dev@tuned"]
    print(f"{n:<24}{r['trainable_M']:>10}M{a['macro_f1']:>8.4f}{b['macro_f1']:>8.4f}{b['t']:>6.2f}"
          f"{b['macro_p']:>7.3f}{b['macro_r']:>7.3f}{b['acc']:>7.3f}{r['test_pos_rate@tuned']:>7.2f}"
          f"{r['infer_test_s']:>8}{r['min']:>6}")

p4_run processes: 0
0, 0 %, 0 MiB
1, 0 %, 0 MiB

--- GPU 0: key events
    08:34:38 === m1_lora_last512: m1 | lora | last-512 | epochs 1 lr 0.0005
    /kaggle/working/src/p4_cjpe.py:206: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite 
    08:42:36    EVAL step 1980: dev macro-F1 0.6385 @0.5 (pos rate 0.65) | 0.6740 @t=0.62 | 475s
    08:50:33    EVAL step 3960: dev macro-F1 0.6715 @0.5 (pos rate 0.29) | 0.6963 @t=0.39 | 951s
    08:51:09 DONE m1_lora_last512: dev macro-F1 0.6715 @0.5 | 0.6963 @t=0.39 | test pos-rate 0.45 | 16.6 min
    08:51:12 === m1_lora_first512: m1 | lora | first-512 | epochs 1 lr 0.0005
    08:59:09    EVAL step 1980: dev macro-F1 0.4714 @0.5 (pos rate 0.84) | 0.5610 @t=0.67 | 477s
    09:07:05    EVAL step 3960: dev macro-F1 0.5515 @0.5 (pos rate 0.56) | 0.5794 @t=0.54 | 953s
    09:07:41 DONE m1_lora_first512: dev macro-F1 0.5515 @0.5 | 0.5794 @t=0.54 | test pos-rate

In [5]:
# ============================================================================================
# P4-B — Backup incl. Phase 4 to the private HF repo + verification
#   Adds the token cache, clean split lists, checkpoints and probabilities of Phase 4 to the
#   allow-list (once). The raw 850 MB Phase-4 json is NOT uploaded: it re-downloads in ~20 s.
# ============================================================================================
import os, subprocess
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi

WORK = "/kaggle/working"
p = f"{WORK}/src/backup_hf.py"

# ---- 1. extend the allow-list (idempotent) -------------------------------------------------------
s = open(p).read()
if "p4_ckpt" not in s:
    s = s.replace('"submissions_p2/**",',
                  '"submissions_p2/**", "data_p4/tok/**", "data_p4/clean/**", "p4_ckpt/**", "p4_prob/**", "submissions_p4/**",')
    open(p, "w").write(s)
print("Phase-4 folders in allow-list:", "p4_ckpt" in open(p).read())

# ---- 2. run the backup (token via environment only) -------------------------------------------
env = {**os.environ, "HF_WRITE_TOKEN": UserSecretsClient().get_secret("HF_WRITE_TOKEN")}
r = subprocess.run(f"python {p}", shell=True, capture_output=True, text=True, env=env)
print(r.stdout[-600:])
if r.returncode != 0:
    print(r.stderr[-1500:])

# ---- 3. verify: Phase-4 files now on the hub -------------------------------------------------
api = HfApi(token=env["HF_WRITE_TOKEN"])
files = api.list_repo_files(f"{api.whoami()['name']}/cs781-phase1-work")
for pre in ["src/p4_", "data_p4/tok/", "data_p4/clean/", "p4_ckpt/", "p4_prob/", "stats/p4_"]:
    hit = [f for f in files if f.startswith(pre)]
    print(f"  {pre:<16} {len(hit):3d} files", ("  e.g. " + hit[0]) if hit else "")

Phase-4 folders in allow-list: True
BACKUP DONE -> Utkarsh-1718-1/cs781-phase1-work in 44s

  src/p4_            2 files   e.g. src/p4_cjpe.py
  data_p4/tok/       3 files   e.g. data_p4/tok/dev.npz
  data_p4/clean/     3 files   e.g. data_p4/clean/dev.json
  p4_ckpt/           4 files   e.g. p4_ckpt/m1_lora_first512.pt
  p4_prob/           4 files   e.g. p4_prob/m1_lora_first512.pkl
  stats/p4_         10 files   e.g. stats/p4_gpu0.log


## P4-4 · Final model and hierarchical model

**P4-3 result (1 epoch, budget-matched):** full FT 0.738 > LoRA on the **last 1,024 tokens** 0.731 > gradual unfreezing 0.730 ≈ partial (top 4) 0.728 > adapters ≈ LoRMA+ ≈ LoRA 0.696–0.702 > norm-gain 0.634. With 31.7k training cases, trainable capacity matters, which is the opposite of the 80-document RR task in Phase 3. Doubling the context of the closing reasoning (512 → 1,024 tokens) adds +0.035.

* **Final model:** M1 full FT on the last 1,024 tokens, 2 epochs.
* **Hierarchical model:** the fine-tuned last-512 model is frozen and encodes every 512-token chunk of each case (at most the last 64). A BiGRU with attention pooling over the chunk sequence predicts the outcome. The attention mass per position decile shows where in the judgment the evidence is.

In [6]:
%%writefile /kaggle/working/src/p4_run.py
"""
p4_run.py -- background runner for Phase-4 (CJPE) experiments, one process per GPU:

    CUDA_VISIBLE_DEVICES=0 PYTHONUNBUFFERED=1 setsid nohup python p4_run.py --plan plan.json >> log 2>&1 &

plan.json = list of jobs. Job types:
  run  (default): {"name", "ckpt": "m1".."m4", "method", "mode": "last", "k": 512, "epochs", "lr",
                   "bs", "seed", "max_train", "layerwise_unfreeze", "partial_k", method kwargs...,
                   KD: "teachers": [run names], "student_layers": [...], "kd_T", "kd_lam"}
  prob:  {"type": "prob", "name": X, "train": true}  re-load run X and save P(1) for train/dev/test
  hier:  {"type": "hier", "name": N, "base": X, "epochs": 8}  hierarchical model: encode EVERY 512-token
         chunk of every case with fine-tuned run X (frozen), then train a BiGRU + attention pooler
         over the chunk sequence; also reports where in the judgment the attention goes
Outputs per run: stats/p4_runs/<name>.json (written last = done marker), p4_ckpt/<name>.pt (trained
params), p4_prob/<name>.pkl ({"dev": P(1), "test": P(1)[, "train"]}).
Model selection / thresholds use the CLEAN dev only (test cases removed); test labels are never read.
"""
import argparse
import json
import os
import pickle
import sys
import time
import traceback

import numpy as np
import torch

WORK = "/kaggle/working"
sys.path.insert(0, f"{WORK}/src")
import p4_cjpe as cj                                                  # noqa: E402
import p2_peft as pp                                                  # noqa: E402
import p3_rr as rr                                                    # noqa: E402

ap = argparse.ArgumentParser(); ap.add_argument("--plan", required=True)
PLAN = json.load(open(ap.parse_args().plan))
DEV = "cuda:0"
CK = {"m1": f"{WORK}/checkpoints/m1_dense_100m/best.pt", "m2": f"{WORK}/checkpoints/m2_mla_moe/best.pt",
      "m3": f"{WORK}/checkpoints/m3_encdec/best.pt", "m4": f"{WORK}/checkpoints/m4_copy/best.pt"}
OUT, PRB, CKD = f"{WORK}/stats/p4_runs", f"{WORK}/p4_prob", f"{WORK}/p4_ckpt"
for d in (OUT, PRB, CKD):
    os.makedirs(d, exist_ok=True)


def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)


TR, DV, TE = (cj.Split(f"{WORK}/data_p4/tok/{s}.npz") for s in ("train", "dev", "test"))
log(f"data: train {TR.n} | clean dev {DV.n} | test {TE.n}")


def make_model(ckpt, method, student_layers=None, partial_k=4, **kw):
    m = cj.build(CK[ckpt], method, bidirectional=True, partial_k=partial_k, **kw)
    if student_layers:
        pp.shrink_dense(m.tagger, tuple(student_layers))
    return m


def run(name, ckpt="m1", method="lora", mode="last", k=512, epochs=1, lr=5e-4, bs=8, seed=0,
        max_train=None, layerwise_unfreeze=False, partial_k=4, teachers=None, student_layers=None,
        kd_T=2.0, kd_lam=0.5, **kw):
    t0 = time.time()
    model = make_model(ckpt, method, student_layers, partial_k, **kw)
    soft = None
    if teachers:                                                       # mean teacher P(1) on train cases
        soft = np.mean([pickle.load(open(f"{PRB}/{t}.pkl", "rb"))["train"] for t in teachers], 0)
    log(f"=== {name}: {ckpt} | {method} | {mode}-{k} | epochs {epochs} lr {lr}"
        + (f" | student layers {student_layers} | teachers {teachers}" if teachers or student_layers else ""))
    hist, best = cj.train(model, TR, DV, mode, k, epochs, lr, bs=bs, device=DEV, seed=seed,
                          max_train=max_train, layerwise_unfreeze=layerwise_unfreeze,
                          soft=soft, kd_T=kd_T, kd_lam=kd_lam, log=log)
    pd = cj.predict(model, DV, mode, k, DEV)
    torch.cuda.synchronize(); ti = time.time()
    pt = cj.predict(model, TE, mode, k, DEV)
    torch.cuda.synchronize(); infer = round(time.time() - ti, 1)
    t = cj.tune_threshold(DV.labels, pd)
    out = {"cfg": dict(ckpt=ckpt, method=method, mode=mode, k=k, epochs=epochs, lr=lr, bs=bs, seed=seed,
                       max_train=max_train, layerwise_unfreeze=layerwise_unfreeze, partial_k=partial_k,
                       teachers=teachers, student_layers=student_layers, kd_T=kd_T, kd_lam=kd_lam, **kw),
           "trainable_M": round(sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6, 3),
           "total_M": round(sum(p.numel() for p in model.parameters()) / 1e6, 2),
           "infer_test_s": infer, "hist": hist, "min": round((time.time() - t0) / 60, 1),
           "dev@0.5": cj.metrics(DV.labels, pd), "dev@tuned": cj.metrics(DV.labels, pd, t),
           "test_pos_rate@tuned": float((pt >= t).mean())}
    torch.save({n: p.detach().half().cpu() for n, p in model.named_parameters()
                if p.requires_grad or layerwise_unfreeze or method in ("full", "partial")}, f"{CKD}/{name}.pt")
    pickle.dump({"dev": pd, "test": pt}, open(f"{PRB}/{name}.pkl", "wb"))
    json.dump(out, open(f"{OUT}/{name}.json", "w"), indent=1)
    log(f"DONE {name}: dev macro-F1 {out['dev@0.5']['macro_f1']:.4f} @0.5 | {out['dev@tuned']['macro_f1']:.4f} "
        f"@t={t:.2f} | test pos-rate {out['test_pos_rate@tuned']:.2f} | {out['min']} min")
    del model; torch.cuda.empty_cache()


def prob_job(name, train=False):
    R = json.load(open(f"{OUT}/{name}.json")); c = {k: v for k, v in R["cfg"].items() if v is not None}
    mode, k = c.pop("mode"), c.pop("k")
    for key in ("epochs", "lr", "bs", "seed", "max_train", "layerwise_unfreeze", "teachers", "kd_T", "kd_lam"):
        c.pop(key, None)
    m = make_model(c.pop("ckpt"), c.pop("method"), c.pop("student_layers", None), c.pop("partial_k", 4), **c)
    sd = {n: v.float() for n, v in torch.load(f"{CKD}/{name}.pt", map_location="cpu").items()}
    m.load_state_dict(sd, strict=False); m.to(DEV)
    path = f"{PRB}/{name}.pkl"; out = pickle.load(open(path, "rb"))
    if train and "train" not in out:
        ti = time.time(); out["train"] = cj.predict(m, TR, mode, k, DEV)
        log(f"   {name}: train probs in {time.time()-ti:.0f}s")
    pickle.dump(out, open(path, "wb"))
    chk = cj.metrics(DV.labels, cj.predict(m, DV, mode, k, DEV))["macro_f1"]
    log(f"PROB {name}: dev re-check {chk:.4f} (should equal its @0.5 score)")
    del m; torch.cuda.empty_cache()


def _load_run(name):
    """Rebuild run `name` from its result file and load its trained weights."""
    R = json.load(open(f"{OUT}/{name}.json")); c = {k: v for k, v in R["cfg"].items() if v is not None}
    for key in ("mode", "k", "epochs", "lr", "bs", "seed", "max_train", "layerwise_unfreeze", "teachers",
                "kd_T", "kd_lam", "type", "base"):
        c.pop(key, None)
    m = make_model(c.pop("ckpt"), c.pop("method"), c.pop("student_layers", None), c.pop("partial_k", 4), **c)
    sd = {n: v.float() for n, v in torch.load(f"{CKD}/{name}.pt", map_location="cpu").items()}
    m.load_state_dict(sd, strict=False)
    return m.to(DEV)


def _pad(E, idx):
    C = max(len(E[i]) for i in idx)
    x = np.zeros((len(idx), C, E[idx[0]].shape[1]), dtype=np.float32); mk = np.zeros((len(idx), C), dtype=bool)
    for b, i in enumerate(idx):
        x[b, :len(E[i])] = E[i]; mk[b, :len(E[i])] = True
    return torch.from_numpy(x).to(DEV), torch.from_numpy(mk).to(DEV)


@torch.no_grad()
def _pool_predict(pool, E, bs=64, return_att=False):
    pool.eval(); n = len(E); p = np.zeros(n, dtype=np.float32); atts = [None] * n
    order = np.argsort([len(e) for e in E])
    for b in range(0, n, bs):
        idx = order[b:b + bs]; x, mk = _pad(E, idx)
        o, a = pool(x, mk, return_att=True)
        p[idx] = torch.softmax(o, -1)[:, 1].cpu().numpy()
        if return_att:
            for j, i in enumerate(idx):
                atts[i] = a[j, :len(E[i])].cpu().numpy()
    return (p, atts) if return_att else p


def hier_job(name, base, epochs=8, lr=1e-3, chunk=512, max_chunks=64, bs=32, seed=0):
    t0 = time.time(); torch.manual_seed(seed); np.random.seed(seed)
    cache = f"{WORK}/p4_chunks/{base}.pkl"; os.makedirs(f"{WORK}/p4_chunks", exist_ok=True)
    if os.path.exists(cache):
        E = pickle.load(open(cache, "rb")); log(f"   loaded chunk embeddings of {base}")
    else:
        m = _load_run(base); E = {}
        for s, sp in (("dev", DV), ("test", TE), ("train", TR)):
            ti = time.time(); E[s] = cj.encode_chunks(m, sp, chunk, DEV, max_chunks=max_chunks)
            log(f"   encoded {s}: {sum(len(e) for e in E[s])} chunks in {time.time()-ti:.0f}s")
        del m; torch.cuda.empty_cache(); pickle.dump(E, open(cache, "wb"))
    pool = cj.ChunkPooler(E["train"][0].shape[1]).to(DEV)
    y = TR.labels; cnt = np.bincount(y, minlength=2).astype(float)
    w = torch.tensor(cnt.sum() / (2 * cnt), dtype=torch.float, device=DEV)
    opt = torch.optim.AdamW(pool.parameters(), lr=lr, weight_decay=0.01)
    steps = epochs * int(np.ceil(TR.n / bs))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps, pct_start=0.1)
    log(f"=== {name}: hierarchical BiGRU+attention over chunks of {base} | {sum(p.numel() for p in pool.parameters())/1e6:.2f}M params")
    best, best_state, hist = -1.0, None, []
    for ep in range(epochs):
        pool.train(); order = np.random.permutation(TR.n); tot = 0.0
        for b in range(0, TR.n, bs):
            idx = order[b:b + bs]; x, mk = _pad(E["train"], idx)
            loss = torch.nn.functional.cross_entropy(pool(x, mk), torch.tensor(y[idx], device=DEV), weight=w)
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(pool.parameters(), 1.0); opt.step(); sched.step(); tot += loss.item()
        pd = _pool_predict(pool, E["dev"]); m05 = cj.metrics(DV.labels, pd)
        mt = cj.metrics(DV.labels, pd, cj.tune_threshold(DV.labels, pd))
        hist.append({"epoch": ep + 1, "f1@0.5": m05["macro_f1"], "f1@tuned": mt["macro_f1"]})
        log(f"   epoch {ep+1}: loss {tot/np.ceil(TR.n/bs):.4f} | dev macro-F1 {m05['macro_f1']:.4f} @0.5 | {mt['macro_f1']:.4f} @t={mt['t']:.2f}")
        if m05["macro_f1"] > best:
            best = m05["macro_f1"]; best_state = {k: v.detach().clone() for k, v in pool.state_dict().items()}
    pool.load_state_dict(best_state)
    pd, att = _pool_predict(pool, E["dev"], return_att=True); pt = _pool_predict(pool, E["test"])
    t = cj.tune_threshold(DV.labels, pd)
    prof, n_docs = np.zeros(10), 0                                    # mean attention mass per position decile
    for a in att:
        if len(a) >= 3:                                               # (cases with >= 3 chunks)
            pos = np.minimum(9, (np.arange(len(a)) / len(a) * 10).astype(int))
            np.add.at(prof, pos, a); n_docs += 1
    out = {"cfg": {"type": "hier", "base": base, "epochs": epochs, "lr": lr, "chunk": chunk, "max_chunks": max_chunks},
           "trainable_M": round(sum(p.numel() for p in pool.parameters()) / 1e6, 3), "total_M": 0.0,
           "infer_test_s": 0.0, "hist": hist, "min": round((time.time() - t0) / 60, 1),
           "dev@0.5": cj.metrics(DV.labels, pd), "dev@tuned": cj.metrics(DV.labels, pd, t),
           "test_pos_rate@tuned": float((pt >= t).mean()),
           "attention_by_position_decile": [round(float(v), 4) for v in prof / max(1, n_docs)]}
    torch.save(pool.state_dict(), f"{CKD}/{name}.pt")
    pickle.dump({"dev": pd, "test": pt}, open(f"{PRB}/{name}.pkl", "wb"))
    json.dump(out, open(f"{OUT}/{name}.json", "w"), indent=1)
    log(f"DONE {name}: dev macro-F1 {out['dev@0.5']['macro_f1']:.4f} @0.5 | {out['dev@tuned']['macro_f1']:.4f} @t={t:.2f} "
        f"| attention by decile {out['attention_by_position_decile']} | {out['min']} min")


for job in PLAN:
    job = dict(job); name = job.pop("name"); typ = job.pop("type", "run")
    try:
        if typ == "prob":
            prob_job(name, job.get("train", False))
        elif typ == "hier":
            if os.path.exists(f"{OUT}/{name}.json"):
                log(f"skip {name} (already done)")
            else:
                hier_job(name, **job)
        elif os.path.exists(f"{OUT}/{name}.json"):
            log(f"skip {name} (already done)")
        else:
            run(name, **job)
    except Exception:
        log(f"!!! {name} FAILED"); traceback.print_exc(); torch.cuda.empty_cache()
log("PLAN DONE")

Overwriting /kaggle/working/src/p4_run.py


In [2]:
# ============================================================================================
# P4-4L (v2) — launch both P4-4 jobs directly (both GPUs are free)
#   GPU 0: final model (M1 full FT, last 1024 tokens, 2 epochs)  ~100 min
#   GPU 1: hierarchical BiGRU+attention over all 512-token chunks of m1_full_last512  ~60 min
# ============================================================================================
import json, subprocess
WORK = "/kaggle/working"
PLANS = {
    0: [dict(name="m1_full_last1024", ckpt="m1", method="full", mode="last", k=1024, epochs=2, lr=3e-5, bs=8)],
    1: [dict(type="hier", name="m1_full512_hier", base="m1_full_last512", epochs=8)],
}
if subprocess.run("pgrep -f '[p]4_run.py'", shell=True, capture_output=True, text=True).stdout.strip():
    print("p4_run.py already running -> not launching again")
else:
    for g, plan in PLANS.items():
        pf = f"{WORK}/stats/p4_plan4_gpu{g}.json"; json.dump(plan, open(pf, "w"))
        subprocess.run(f"cd {WORK}/src && CUDA_VISIBLE_DEVICES={g} PYTHONUNBUFFERED=1 setsid nohup "
                       f"python p4_run.py --plan {pf} >> {WORK}/stats/p4_gpu{g}.log 2>&1 < /dev/null &", shell=True)
    print("launched P4-4 on both GPUs -> check with P4-W in ~3 min")

launched P4-4 on both GPUs -> check with P4-W in ~3 min


In [5]:
# ============================================================================================
# P4-W — Phase-4 progress + results (reads logs and result files only)
# ============================================================================================
import json, glob, os, subprocess
WORK = "/kaggle/working"

# ---- 1. background jobs alive? GPUs busy? ---------------------------------------------------------
print(subprocess.run("echo 'p4_run processes:' $(pgrep -fc '[p]4_run.py'); "
                     "nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout)

# ---- 2. per GPU: key events of the CURRENT plan (from the last 'data:' line on) + last lines ----
for g in (0, 1):
    p = f"{WORK}/stats/p4_gpu{g}.log"
    lines = open(p).read().splitlines() if os.path.exists(p) else []
    start = max([i for i, l in enumerate(lines) if "data:" in l] or [0])     # newest launch only
    print(f"--- GPU {g}: key events (current plan)")
    for l in lines[start:]:
        if any(k in l for k in ("===", "EVAL", "encoded", "epoch ", "DONE", "PROB", "FAILED", "Error", "skip")):
            print("   ", l[:200])
    print(f"--- GPU {g}: last 2 lines")
    for l in lines[-2:]:
        print("   ", l[:200])

# ---- 3. results table (all Phase-4 runs), sorted by dev macro-F1 at the tuned threshold ---------
R = {os.path.basename(f)[:-5]: json.load(open(f)) for f in glob.glob(f"{WORK}/stats/p4_runs/*.json")}
print(f"\n{'run':<24}{'trainable':>11}{'F1@0.5':>8}{'F1@t':>8}{'t':>6}{'P':>7}{'R':>7}{'acc':>7}"
      f"{'test+':>7}{'test s':>8}{'min':>6}")
for n, r in sorted(R.items(), key=lambda kv: -kv[1]["dev@tuned"]["macro_f1"]):
    a, b = r["dev@0.5"], r["dev@tuned"]
    print(f"{n:<24}{r['trainable_M']:>10}M{a['macro_f1']:>8.4f}{b['macro_f1']:>8.4f}{b['t']:>6.2f}"
          f"{b['macro_p']:>7.3f}{b['macro_r']:>7.3f}{b['acc']:>7.3f}{r['test_pos_rate@tuned']:>7.2f}"
          f"{r['infer_test_s'] or 0:>8}{r['min']:>6}")

# ---- 4. hierarchical model: where in the judgment does the attention go? -------------------------
for n, r in R.items():
    if "attention_by_position_decile" in r:
        prof = r["attention_by_position_decile"]
        print(f"\n{n}: mean attention mass per position decile (0 = start of judgment, 9 = end)")
        for i, v in enumerate(prof):
            print(f"   decile {i}: {v:.3f} {'#' * int(round(v * 100))}")

p4_run processes: 1
0, 99 %, 8073 MiB
1, 0 %, 0 MiB

--- GPU 0: key events (current plan)
    13:57:04 === m1_full_last1024: m1 | full | last-1024 | epochs 2 lr 3e-05
    14:17:48    EVAL step 1980: dev macro-F1 0.6546 @0.5 (pos rate 0.72) | 0.7308 @t=0.71 | 1242s
--- GPU 0: last 2 lines
    14:28:11    step 3000/7920 | loss 0.5341 | 1865s
    14:33:16    step 3500/7920 | loss 0.5284 | 2170s
--- GPU 1: key events (current plan)
    13:57:06 === m1_full512_hier: hierarchical BiGRU+attention over chunks of m1_full_last512 | 1.58M params
    13:57:16    epoch 1: loss 0.4373 | dev macro-F1 0.7852 @0.5 | 0.7906 @t=0.54
    13:57:25    epoch 2: loss 0.4150 | dev macro-F1 0.7660 @0.5 | 0.7867 @t=0.21
    13:57:34    epoch 3: loss 0.3987 | dev macro-F1 0.7818 @0.5 | 0.7871 @t=0.30
    13:57:43    epoch 4: loss 0.3757 | dev macro-F1 0.7774 @0.5 | 0.7928 @t=0.38
    13:57:51    epoch 5: loss 0.3428 | dev macro-F1 0.7736 @0.5 | 0.7885 @t=0.21
    13:57:59    epoch 6: loss 0.2954 | dev macro-F1 0.

In [1]:
# ============================================================================================
# R5 — After a session stop: what survived locally, and what is on HF?
# ============================================================================================
import os, glob
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi
W = "/kaggle/working"
print("code present:   ", all(os.path.exists(f"{W}/src/{f}") for f in ["p4_cjpe.py", "p4_run.py", "p3_rr.py", "p2_peft.py"]))
print("token cache:    ", sorted(os.listdir(f"{W}/data_p4/tok")) if os.path.exists(f"{W}/data_p4/tok") else "MISSING")
print("finished runs:  ", sorted(os.path.basename(f)[:-5] for f in glob.glob(f"{W}/stats/p4_runs/*.json")))
print("chunk cache:    ", os.listdir(f"{W}/p4_chunks") if os.path.exists(f"{W}/p4_chunks") else "none")
for g in (0, 1):                                   # last lines of each log -> how far the jobs got
    p = f"{W}/stats/p4_gpu{g}.log"
    if os.path.exists(p):
        print(f"--- GPU {g} log tail:"); [print("   ", l[:160]) for l in open(p).read().splitlines()[-3:]]
api = HfApi(token=UserSecretsClient().get_secret("HF_WRITE_TOKEN"))
hf = [f for f in api.list_repo_files(f"{api.whoami()['name']}/cs781-phase1-work") if f.startswith("stats/p4_runs/")]
print("finished runs on HF:", len(hf))

code present:    True
token cache:     ['dev.npz', 'test.npz', 'train.npz']
finished runs:   ['m1_adapter_last512', 'm1_full_last512', 'm1_lora_first512', 'm1_lora_headtail512', 'm1_lora_last1024', 'm1_lora_last512', 'm1_lora_middle512', 'm1_lorma_plus_last512', 'm1_normgain_last512', 'm1_partial4_last512', 'm1_unfreeze_last512', 'm2_lorma_plus_last512', 'm3_full_last512', 'm4_adapter_last512']
chunk cache:     ['m1_full_last512.pkl']
--- GPU 0 log tail:
    23:22:37    step 2500/7920 | loss 0.5515 | 1698s
    23:28:11    step 3000/7920 | loss 0.5327 | 2032s
    23:33:43    step 3500/7920 | loss 0.5275 | 2364s
--- GPU 1 log tail:
    23:38:09    epoch 5: loss 0.3397 | dev macro-F1 0.7778 @0.5 | 0.7947 @t=0.29
    23:38:18    epoch 6: loss 0.2919 | dev macro-F1 0.7795 @0.5 | 0.7925 @t=0.34
    23:38:27    epoch 7: loss 0.2456 | dev macro-F1 0.7739 @0.5 | 0.7897 @t=0.18
finished runs on HF: 4


In [ ]:
# P4-B (v2) — backup incl. all Phase-4 runs AND the chunk-embedding cache (~0.3-0.5 GB)
import os, subprocess
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi
WORK = "/kaggle/working"; p = f"{WORK}/src/backup_hf.py"
s = open(p).read()
if "p4_ckpt" not in s:                                               # Phase-4 folders (first time)
    s = s.replace('"submissions_p2/**",', '"submissions_p2/**", "data_p4/tok/**", "data_p4/clean/**", '
                  '"p4_ckpt/**", "p4_prob/**", "submissions_p4/**",')
if "p4_chunks" not in s:                                             # chunk cache (new)
    s = s.replace('"p4_ckpt/**",', '"p4_ckpt/**", "p4_chunks/**",')
open(p, "w").write(s)
env = {**os.environ, "HF_WRITE_TOKEN": UserSecretsClient().get_secret("HF_WRITE_TOKEN")}
r = subprocess.run(f"python {p}", shell=True, capture_output=True, text=True, env=env)
print(r.stdout[-400:]); print(r.stderr[-800:] if r.returncode else "")
api = HfApi(token=env["HF_WRITE_TOKEN"])
files = api.list_repo_files(f"{api.whoami()['name']}/cs781-phase1-work")
for pre in ["stats/p4_runs/", "p4_ckpt/", "p4_prob/", "p4_chunks/", "data_p4/tok/"]:
    print(f"  {pre:<16} {sum(f.startswith(pre) for f in files):3d} files")

## P4-5 · Knowledge distillation

**Teacher:** M1 full FT on the last 512 tokens (100M parameters; its predicted probabilities on the training cases are the soft targets). **Student:** M1 with 6 of 12 layers (62M parameters), full FT on the last 512 tokens, loss = 0.5·CE + 0.5·T²·KL(teacher ‖ student), T = 2, 1 epoch. **Control:** the same student without the teacher. We compare macro-F1, parameter count and test inference time.

In [7]:
# ============================================================================================
# P4-5L — KD on GPU 1 (GPU 0 keeps training the final model)
#   teacher = m1_full_last512 (train-set P(1) first) -> 6-layer M1 student; + no-KD control
#   Safe to re-run: refuses if a GPU-1 job is running; finished runs are skipped by the runner.
# ============================================================================================
import json, subprocess
WORK = "/kaggle/working"
STUDENT = dict(ckpt="m1", method="full", mode="last", k=512, epochs=1, lr=5e-5, bs=8,
               student_layers=[0, 2, 4, 6, 8, 10])
plan = [dict(type="prob", name="m1_full_last512", train=True),                  # teacher probs on train
        dict(name="kd_student_m1x6_last512", teachers=["m1_full_last512"], kd_T=2.0, kd_lam=0.5, **STUDENT),
        dict(name="nokd_student_m1x6_last512", **STUDENT)]                       # same student, no teacher
pf = f"{WORK}/stats/p4_plan5_gpu1.json"
if subprocess.run("pgrep -f '[p]4_run.py --plan .*_gpu1.json'", shell=True, capture_output=True, text=True).stdout.strip():
    print("a GPU-1 job is already running -> not launching")
else:
    json.dump(plan, open(pf, "w"))
    subprocess.run(f"cd {WORK}/src && CUDA_VISIBLE_DEVICES=1 PYTHONUNBUFFERED=1 setsid nohup "
                   f"python p4_run.py --plan {pf} >> {WORK}/stats/p4_gpu1.log 2>&1 < /dev/null &", shell=True)
    print("launched KD on GPU 1")

launched KD on GPU 1


In [8]:
# ============================================================================================
# P4-Q — Block until no Phase-4 background job is running (heartbeat every 5 min)
# ============================================================================================
import subprocess, time, os
def n_jobs():
    out = subprocess.run("pgrep -fc '[p]4_run.py'", shell=True, capture_output=True, text=True).stdout.strip()
    return int(out or 0)
while n_jobs() > 0:
    tails = []
    for g in (0, 1):                                                  # last log line per GPU
        p = f"/kaggle/working/stats/p4_gpu{g}.log"
        if os.path.exists(p):
            tails.append(f"GPU{g}: " + open(p).read().splitlines()[-1][:110])
    print(time.strftime("%H:%M"), f"| {n_jobs()} job(s) running |", " || ".join(tails), flush=True)
    time.sleep(300)
print(time.strftime("%H:%M"), "all Phase-4 background jobs finished")

15:00 | 2 job(s) running | GPU0: 15:00:12    step 6000/7920 | loss 0.4036 | 3786s || GPU1: 14:59:28 data: train 31677 | clean dev 1988 | test 1517
15:05 | 2 job(s) running | GPU0: 15:05:18    step 6500/7920 | loss 0.3953 | 4092s || GPU1: 15:05:13 trainable 62.337M | 31677 cases | 3960 steps | view last-512 | KD
15:10 | 2 job(s) running | GPU0: 15:05:18    step 6500/7920 | loss 0.3953 | 4092s || GPU1: 15:09:28    step 1500/3960 | loss 0.3400 | 255s
15:15 | 2 job(s) running | GPU0: 15:10:24    step 7000/7920 | loss 0.3813 | 4397s || GPU1: 15:13:56    step 3000/3960 | loss 0.2878 | 523s
15:20 | 2 job(s) running | GPU0: 15:15:29    step 7500/7920 | loss 0.3682 | 4703s || GPU1: 15:20:04    step 1000/3960 | loss 0.6559 | 170s
15:25 | 1 job(s) running | GPU0: 15:21:49 PLAN DONE || GPU1: 15:24:30    step 2500/3960 | loss 0.5666 | 435s
15:30 all Phase-4 background jobs finished


## P4-6 · More hierarchical models   ·   P4-7 · Ensembles, error analysis, submissions

**Hierarchical result (P4-4):** a BiGRU + attention pooler over all 512-token chunks of each case (encoder: frozen M1 fine-tuned on the last 512 tokens) reaches **0.791** dev macro-F1, against 0.738 for the best last-512 model (+0.053). Attention mass per decile of the judgment: 73% falls on the last 40%, peaking at decile 8 (0.30), just before the removed decision. The opening deciles get little attention, apart from the case header.

**P4-6:** the same hierarchy on (a) the final last-1,024 model, with 1,024-token chunks, and (b) the partial-FT model (a different backbone, for ensemble diversity).

**P4-7 (CPU):** single models and greedy ensembles (mean probability) on clean dev, with the threshold tuned on clean dev; error analysis (false positives vs false negatives, macro-F1 by case length, calibration); test submissions `[{"id", "pred"}]`.

In [9]:
# ============================================================================================
# P4-6L — Hierarchical models on two more backbones (one per GPU, ~60 min each)
#   GPU 0: final M1 full-FT last-1024 model, 1024-token chunks (<= 32 per case = last 32k tokens)
#   GPU 1: M1 partial-FT (top-4 layers) last-512 model, 512-token chunks (ensemble diversity)
# ============================================================================================
import json, subprocess, os
WORK = "/kaggle/working"
PLANS = {0: [dict(type="hier", name="m1_full1024_hier", base="m1_full_last1024", chunk=1024, max_chunks=32, epochs=8)],
         1: [dict(type="hier", name="m1_partial512_hier", base="m1_partial4_last512", chunk=512, max_chunks=64, epochs=8)]}
if not os.path.exists(f"{WORK}/stats/p4_runs/m1_full_last1024.json"):
    print("!! final model m1_full_last1024 not finished -> run P4-Q first")
elif subprocess.run("pgrep -f '[p]4_run.py'", shell=True, capture_output=True, text=True).stdout.strip():
    print("p4_run.py already running -> not launching")
else:
    for g, plan in PLANS.items():
        pf = f"{WORK}/stats/p4_plan6_gpu{g}.json"; json.dump(plan, open(pf, "w"))
        subprocess.run(f"cd {WORK}/src && CUDA_VISIBLE_DEVICES={g} PYTHONUNBUFFERED=1 setsid nohup "
                       f"python p4_run.py --plan {pf} >> {WORK}/stats/p4_gpu{g}.log 2>&1 < /dev/null &", shell=True)
    print("launched 2 hierarchical models")

launched 2 hierarchical models


In [2]:
# ============================================================================================
# P4-7 — Single models + greedy ensembles (clean dev), error analysis, test submissions
#   Uses only saved probabilities (p4_prob/*.pkl); thresholds tuned on CLEAN dev; no test labels used.
# ============================================================================================
import os, sys, json, glob, pickle, zipfile
import numpy as np
WORK = "/kaggle/working"; sys.path.insert(0, f"{WORK}/src")
import p4_cjpe as cj

DV = cj.Split(f"{WORK}/data_p4/tok/dev.npz"); y = DV.labels
dev_len = np.diff(DV.off)                                            # tokens per dev case
test_ids = [r["id"] for r in json.load(open(f"{WORK}/data_p4/clean/test.json"))]
P = {os.path.basename(f)[:-4]: pickle.load(open(f, "rb")) for f in glob.glob(f"{WORK}/p4_prob/*.pkl")}
assert all(len(P[n]["test"]) == len(test_ids) for n in P), "test length mismatch"

def score(p):                                                        # tuned threshold on clean dev
    t = cj.tune_threshold(y, p); return cj.metrics(y, p, t)["macro_f1"], t

# ---- 1. single models ------------------------------------------------------------------------
single = {n: score(P[n]["dev"]) for n in P}
print(f"{'model':<28}{'dev F1@t':>9}{'t':>6}")
for n, (f, t) in sorted(single.items(), key=lambda kv: -kv[1][0]):
    print(f"{n:<28}{f:>9.4f}{t:>6.2f}")

# ---- 2. greedy forward ensemble (mean of P(1)); add a member only if dev F1 improves > 0.002 --
pool = [n for n in P if single[n][0] >= 0.68]                        # reasonably strong candidates only
ens = [max(pool, key=lambda n: single[n][0])]; best = single[ens[0]][0]
while True:
    cand = [(score(np.mean([P[m]["dev"] for m in ens + [n]], 0))[0], n) for n in pool if n not in ens]
    if not cand: break
    f, n = max(cand)
    if f <= best + 0.002 or len(ens) >= 5: break
    ens.append(n); best = f
hier = [n for n in P if n.endswith("_hier")]
ENS = {"ens_greedy": ens, "ens_hier": hier}
print("\nensembles:")
for e, m in ENS.items():
    if m:
        f, t = score(np.mean([P[x]["dev"] for x in m], 0)); print(f"   {e:<12} dev F1 {f:.4f} @t={t:.2f}  members {m}")

# ---- 3. error analysis of the best system -----------------------------------------------------
best_name = max([*ENS] + list(P), key=lambda s: score(np.mean([P[x]["dev"] for x in ENS[s]], 0) if s in ENS else P[s]["dev"])[0])
pb = np.mean([P[x]["dev"] for x in ENS[best_name]], 0) if best_name in ENS else P[best_name]["dev"]
f, t = score(pb); yh = (pb >= t).astype(int)
fp, fn = ((yh == 1) & (y == 0)), ((yh == 0) & (y == 1))
print(f"\nbest system: {best_name}  dev macro-F1 {f:.4f} @t={t:.2f}")
print(f"   false positives {fp.sum()} ({fp.sum()/(y==0).sum():.1%} of true 0) | false negatives {fn.sum()} ({fn.sum()/(y==1).sum():.1%} of true 1)")
print(f"   median length (tokens): correct {int(np.median(dev_len[yh==y]))} | FP {int(np.median(dev_len[fp]))} | FN {int(np.median(dev_len[fn]))}")
q = np.quantile(dev_len, [0, .25, .5, .75, 1])
for i in range(4):                                                   # macro-F1 by case-length quartile
    m = (dev_len >= q[i]) & (dev_len <= q[i + 1])
    print(f"   length {int(q[i]):>6}-{int(q[i+1]):<6} tokens: macro-F1 {cj.metrics(y[m], pb[m], t)['macro_f1']:.4f} (n={m.sum()})")
bins = np.linspace(0, 1, 6)                                          # calibration: confidence vs accuracy
for lo, hi in zip(bins[:-1], bins[1:]):
    m = (pb >= lo) & (pb < hi + (hi == 1))
    if m.sum():
        print(f"   P(1) in [{lo:.1f},{hi:.1f}): n={m.sum():4d} | fraction truly 1 = {y[m].mean():.2f}")

# ---- 4. test submissions: [{"id", "pred"}] with each system's dev-tuned threshold --------------
def write(name, members):
    pd = np.mean([P[x]["dev"] for x in members], 0); pt = np.mean([P[x]["test"] for x in members], 0)
    f, t = score(pd); pred = [{"id": i, "pred": int(v >= t)} for i, v in zip(test_ids, pt)]
    folder = f"{WORK}/submissions_p4/p4_{name}"; os.makedirs(folder, exist_ok=True)
    json.dump(pred, open(f"{folder}/prediction.json", "w"))
    with zipfile.ZipFile(f"{folder}/p4_{name}.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{folder}/prediction.json", arcname="prediction.json")
    return f, t, float(np.mean([r["pred"] for r in pred]))

SUBS = {**{e: m for e, m in ENS.items() if m}, **{n: [n] for n in P}}
summary = {}
print(f"\n{'submission':<30}{'dev F1':>8}{'t':>6}{'test pos-rate':>15}")
for name, members in SUBS.items():
    f, t, pr = write(name, members); summary[name] = {"members": members, "dev_f1": round(f, 4), "t": t, "test_pos_rate": round(pr, 3)}
    print(f"p4_{name:<27}{f:>8.4f}{t:>6.2f}{pr:>15.2f}")
json.dump(summary, open(f"{WORK}/stats/p4_submissions.json", "w"), indent=1)
print(f"\nwrote {len(SUBS)} zips to submissions_p4/ (each prediction.json = list of {len(test_ids)} {{id, pred}})")

model                        dev F1@t     t
m1_full1024_hier               0.8058  0.30
m1_full512_hier                0.7906  0.54
m1_partial512_hier             0.7786  0.45
m1_full_last1024               0.7759  0.54
m1_full_last512                0.7383  0.44
nokd_student_m1x6_last512      0.7334  0.35
m1_lora_last1024               0.7312  0.38
m1_unfreeze_last512            0.7300  0.45
m1_partial4_last512            0.7276  0.48
kd_student_m1x6_last512        0.7227  0.51
m4_adapter_last512             0.7020  0.48
m1_adapter_last512             0.6981  0.37
m1_lorma_plus_last512          0.6975  0.41
m1_lora_last512                0.6963  0.39
m2_lorma_plus_last512          0.6845  0.46
m1_lora_headtail512            0.6718  0.48
m1_normgain_last512            0.6338  0.43
m3_full_last512                0.6231  0.48
m1_lora_middle512              0.5862  0.47
m1_lora_first512               0.5794  0.54

ensembles:
   ens_greedy   dev F1 0.8189 @t=0.42  members ['m1_full1024_hie

In [3]:
# ============================================================================================
# P4-B (final) — Backup of everything Phase 4 (runs, checkpoints, probabilities, chunk caches,
#   submission zips) to the private HF repo, with one automatic retry + verification
# ============================================================================================
import os, subprocess, time
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi

WORK = "/kaggle/working"; p = f"{WORK}/src/backup_hf.py"

# ---- 1. allow-list: Phase-4 folders + chunk caches (idempotent) ---------------------------------
s = open(p).read()
if "p4_ckpt" not in s:
    s = s.replace('"submissions_p2/**",', '"submissions_p2/**", "data_p4/tok/**", "data_p4/clean/**", '
                  '"p4_ckpt/**", "p4_prob/**", "submissions_p4/**",')
if "p4_chunks" not in s:
    s = s.replace('"p4_ckpt/**",', '"p4_ckpt/**", "p4_chunks/**",')
open(p, "w").write(s)

# ---- 2. run the backup; retry once if a growing log file made the upload fail -----------------
env = {**os.environ, "HF_WRITE_TOKEN": UserSecretsClient().get_secret("HF_WRITE_TOKEN")}
for attempt in (1, 2):
    r = subprocess.run(f"python {p}", shell=True, capture_output=True, text=True, env=env)
    print(f"attempt {attempt}:", r.stdout.strip()[-300:])
    if "BACKUP DONE" in r.stdout:
        break
    print(r.stderr[-600:]); time.sleep(20)

# ---- 3. verify what is on the hub ---------------------------------------------------------------
api = HfApi(token=env["HF_WRITE_TOKEN"])
files = api.list_repo_files(f"{api.whoami()['name']}/cs781-phase1-work")
for pre in ["src/p4_", "data_p4/tok/", "stats/p4_runs/", "p4_ckpt/", "p4_prob/", "p4_chunks/", "submissions_p4/"]:
    print(f"  {pre:<16} {sum(f.startswith(pre) for f in files):3d} files")
zips = sorted(f.split("/")[1] for f in files if f.startswith("submissions_p4/") and f.endswith(".zip"))
print(f"\nPhase-4 submission zips on HF ({len(zips)}):", zips)

attempt 1: BACKUP DONE -> Utkarsh-1718-1/cs781-phase1-work in 56s
  src/p4_            2 files
  data_p4/tok/       3 files
  stats/p4_runs/    20 files
  p4_ckpt/          20 files
  p4_prob/          20 files
  p4_chunks/         3 files
  submissions_p4/   44 files

Phase-4 submission zips on HF (22): ['p4_ens_greedy', 'p4_ens_hier', 'p4_kd_student_m1x6_last512', 'p4_m1_adapter_last512', 'p4_m1_full1024_hier', 'p4_m1_full512_hier', 'p4_m1_full_last1024', 'p4_m1_full_last512', 'p4_m1_lora_first512', 'p4_m1_lora_headtail512', 'p4_m1_lora_last1024', 'p4_m1_lora_last512', 'p4_m1_lora_middle512', 'p4_m1_lorma_plus_last512', 'p4_m1_normgain_last512', 'p4_m1_partial4_last512', 'p4_m1_partial512_hier', 'p4_m1_unfreeze_last512', 'p4_m2_lorma_plus_last512', 'p4_m3_full_last512', 'p4_m4_adapter_last512', 'p4_nokd_student_m1x6_last512']


In [1]:
# ============================================================================================
# Did P4-6L run? (plan files written? results present? what do the logs say?)
# ============================================================================================
import os, subprocess
W = "/kaggle/working"
NAMES = {"m1_full1024_hier": 0, "m1_partial512_hier": 1}              # job -> GPU it was planned on

print("p4_run processes running now:",
      subprocess.run("pgrep -fc '[p]4_run.py'", shell=True, capture_output=True, text=True).stdout.strip())
for n, g in NAMES.items():
    plan = os.path.exists(f"{W}/stats/p4_plan6_gpu{g}.json")          # written only if P4-6L launched
    done = os.path.exists(f"{W}/stats/p4_runs/{n}.json")              # written only when the job finished
    cache = [f for f in os.listdir(f"{W}/p4_chunks")] if os.path.exists(f"{W}/p4_chunks") else []
    print(f"\n{n}: launched={plan} | finished={done} | chunk caches present: {cache}")
    log = open(f"{W}/stats/p4_gpu{g}.log").read().splitlines() if os.path.exists(f"{W}/stats/p4_gpu{g}.log") else []
    hits = [i for i, l in enumerate(log) if n in l]                   # every log line mentioning this job
    if hits:
        for l in log[hits[0]: hits[0] + 6] + log[hits[-1]: hits[-1] + 1]:
            print("   ", l[:170])
        tb = [l for l in log[hits[0]:] if "Error" in l or "FAILED" in l]
        if tb:
            print("   !! error lines:", *tb[-3:], sep="\n      ")
    else:
        print("    (no log line mentions this job -> it never started)")

p4_run processes running now: 0

m1_full1024_hier: launched=True | finished=True | chunk caches present: ['m1_full_last512.pkl', 'm1_partial4_last512.pkl', 'm1_full_last1024.pkl']
    16:23:58 === m1_full1024_hier: hierarchical BiGRU+attention over chunks of m1_full_last1024 | 1.58M params
    16:24:06    epoch 1: loss 0.3341 | dev macro-F1 0.7816 @0.5 | 0.7872 @t=0.26
    16:24:13    epoch 2: loss 0.3087 | dev macro-F1 0.7882 @0.5 | 0.8098 @t=0.36
    16:24:20    epoch 3: loss 0.3000 | dev macro-F1 0.7937 @0.5 | 0.8059 @t=0.64
    16:24:27    epoch 4: loss 0.2759 | dev macro-F1 0.7891 @0.5 | 0.8063 @t=0.32
    16:24:34    epoch 5: loss 0.2393 | dev macro-F1 0.7947 @0.5 | 0.8058 @t=0.30
    16:24:56 DONE m1_full1024_hier: dev macro-F1 0.7947 @0.5 | 0.8058 @t=0.30 | attention by decile [0.0402, 0.0062, 0.0339, 0.1213, 0.018, 0.0949, 0.238, 0.1728, 0.2249, 0.0

m1_partial512_hier: launched=True | finished=True | chunk caches present: ['m1_full_last512.pkl', 'm1_partial4_last512.pkl', 'm1